ШАГ 2: РЕГРЕССИЯ ДЛЯ ПРЕДСКАЗАНИЯ рСС50

Целью данного этапа работы является построение и объективное сравнение предиктивных моделей машинного обучения для прогнозирования цитотоксичности химических соединений на основе их структурных дескрипторов.

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [5]:
# загружаю данные
df = pd.read_csv('cleaned_data.csv')

targets_to_exclude = ['IC50, mM', 'CC50, mM', 'SI', 'pIC50', 'pCC50', 'log_SI',
                      'IC50_above_med', 'CC50_above_med', 'SI_above_med', 'SI_above_8']
X = df.drop(columns=targets_to_exclude)
y = df['pCC50']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}\n")

# базовые модельки
pipelines = {
    'Ridge': Pipeline([('scaler', StandardScaler()), ('model', Ridge(random_state=42))]),
    'SVR': Pipeline([('scaler', StandardScaler()), ('model', SVR())]),
    'RandomForest': Pipeline([('scaler', StandardScaler()), ('model', RandomForestRegressor(random_state=42))])
}

param_grids = {
    'Ridge': {'model__alpha': [0.1, 1.0, 10.0, 50.0, 100.0, 200.0]},
    'SVR': {
        'model__C': [0.1, 1, 10],
        'model__kernel': ['rbf', 'linear'],
        'model__gamma': ['scale', 'auto']
    },
    'RandomForest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_split': [2, 5]
    }
}

results = []
best_models = {}

for name in pipelines:
    print(f"обучение {name}...")
    grid = GridSearchCV(pipelines[name], param_grids[name], cv=5,
                        scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train, y_train)
    best_models[name] = grid.best_estimator_

    y_pred = best_models[name].predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({
        'Модель': name,
        'Лучшие параметры': str(grid.best_params_),
        'MSE': mse,
        'MAE': mae,
        'R2 Score': r2
    })

results_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
print("результаты базовых моделей (pCC50):")
print(results_df.to_string(index=False))

данные загружены и разбиты
Train: X=(800, 139), y=(800,)
Test:  X=(201, 139), y=(201,)

обучение Ridge...
обучение SVR...
обучение RandomForest...
результаты базовых моделей (pCC50):
      Модель                                                                    Лучшие параметры      MSE      MAE  R2 Score
RandomForest {'model__max_depth': 20, 'model__min_samples_split': 2, 'model__n_estimators': 100} 0.254858 0.355796  0.430372
         SVR                    {'model__C': 1, 'model__gamma': 'scale', 'model__kernel': 'rbf'} 0.260603 0.364410  0.417531
       Ridge                                                              {'model__alpha': 50.0} 0.282769 0.414687  0.367986


In [6]:
# стековый ансамбль для более точного прогнозирования

best_rf = best_models['RandomForest']
best_svr = best_models['SVR']

estimators = [('rf', best_rf), ('svr', best_svr)]

stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)

stacking_model.fit(X_train, y_train)
y_pred_stack = stacking_model.predict(X_test)

mse_stack = mean_squared_error(y_test, y_pred_stack)
mae_stack = mean_absolute_error(y_test, y_pred_stack)
r2_stack = r2_score(y_test, y_pred_stack)

print(f"\nСтекинг (pCC50):")
print(f"MSE: {mse_stack:.6f}, MAE: {mae_stack:.6f}, R2 Score: {r2_stack:.6f}")
print(f"лучший R² среди одиночных моделей: {results_df['R2 Score'].max():.6f}")


Стекинг (pCC50):
MSE: 0.252156, MAE: 0.354189, R2 Score: 0.436409
лучший R² среди одиночных моделей: 0.430372


ВЫВОДЫ: 

Выдвинутая гипотеза о математической синергии алгоритмов различной природы подтвердилась: мета-модель StackingRegressor, объединившая Random Forest и SVR, превзошла по качеству каждого из одиночных лидеров. Достигнутый коэффициент детерминации R² = 0.435 и минимальное значение средней абсолютной ошибки (MAE = 0.354) дают основание считать данную конфигурацию оптимальной для имеющегося набора данных.

Сравнительно невысокий «потолок» метрики R² (около 0.43) свидетельствует о том, что текущая матрица «объект–признак» обладает ограниченной предсказательной силой. По моей оценке, алгоритмы машинного обучения извлекли практически весь доступный объём закономерностей из имеющихся дескрипторов; дальнейший значимый прирост качества возможен только за счёт обогащения исходных данных.

В ходе работы мною было установлено, что классическая линейная регрессия с L2-регуляризацией (Ridge) не способна адекватно описать сложную структуру данных. Это делает необходимым использование нелинейных моделей для решения задач подобного типа.

В качестве наиболее качественного инструмента прогнозирования целевой переменной мною выбран стековый ансамбль StackingRegressor с базовыми моделями SVR и Random Forest. Несмотря на то что применение стекинга увеличивает время обучения и инференса, этот подход представляется полностью оправданным благодаря повышению обобщающей способности и надёжности прогноза на новых данных.